# CC1π systematics

In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp
import ROOT
from ROOT import TMVA

from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
import pandas as pd
import numpy as np
import matplotlib as mpl
from os import path
import sys
import uproot
from tqdm import tqdm

import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

import pyanalib.pandas_helpers as ph
# import dunestyle.matplotlib as dunestyle
#plt.style.use("presentation.mplstyle")

print('Importing Helper Funcions...')
# Add the directory containing Constants to the system path
if ('./HelperFunctions' not in sys.path):
    sys.path.append('./HelperFunctions')  # project root
# Import the constants
import HelperFunctions

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")

# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_fig = False
save_fig_dir = ""

In [ ]:
cmap = mpl.cm.viridis
norm = mpl.colors.Normalize(vmin=0.0, vmax=1.0)

# load dataframes

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_5e18_CV.df", keys2load, 100)
mc_evt_df = mc_bnb_df['cc1pi']
mc_nu_df = mc_bnb_df['nudf']
mc_hdr_df = mc_bnb_df['hdr']

#Add weight column
data_tot_pot = 5.947e+18
mc_tot_pot = mc_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_evt_df))

#Do truth matchign
mc_evt_df = perform_truth_matching(mc_evt_df, mc_nu_df)
mc_nu_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_nu_df))

In [ ]:
mc_obvious_cosmic_mask = mc_evt_df.slc.cut.obvious_cosmic
mc_t0_mask = mc_evt_df.slc.cut.t0
mc_is_inside_FV_mask = mc_evt_df.slc.cut.inside_FV
mc_nu_score_mask = mc_evt_df.slc.cut.nu_score
mc_track_mask = mc_evt_df.slc.cut.track
mc_shower_mask = mc_evt_df.slc.cut.shower 
mc_chi2_mask = mc_evt_df.slc.cut.MIP_candidates 
mc_angle_mask = mc_evt_df.slc.cut.angle 
mc_proton_BDT_mask = mc_evt_df.slc.cut.proton_BDT
mc_containment_mask = mc_evt_df.slc.cut.containment 
mc_michel_mask = mc_evt_df.slc.cut.michel 
mc_extra_pion_mask = mc_evt_df.slc.cut.extra_pion 
mc_energy_mask = mc_evt_df.slc.cut.energy

# 1. Define the order of cuts
mc_cut_sequence = [
    ("cosmic", mc_obvious_cosmic_mask),
    ("t0", mc_t0_mask),
    ("FV", mc_is_inside_FV_mask),
    ("nu_score", mc_nu_score_mask),
    ("track", mc_track_mask),
    ("chi2", mc_chi2_mask),
    ("shower", mc_shower_mask),
    ("angle", mc_angle_mask),
    ("proton_BDT", mc_proton_BDT_mask),
    ("containment", mc_containment_mask),
    ("michel", mc_michel_mask),
    ("extra_pion", mc_extra_pion_mask),
    ("energy", mc_energy_mask)
]

# 2. Build the cumulative masks
mc_cumulative_mak = None

for name, mask in mc_cut_sequence:
    if mc_cumulative_mak is None:
        mc_cumulative_mak = mask
    else:
        mc_cumulative_mak = mc_cumulative_mak & mask

In [ ]:
mc_evt_df = mc_evt_df[mc_cumulative_mak]

In [ ]:
mc_evt_df = (
        mc_evt_df
        .groupby(['__ntuple', 'entry', 'rec.slc..index'])
        .first()
    )
mc_evt_df = mc_evt_df.sort_index()

# Set up utils and selections according to target channel

In [ ]:
mc_evt_df.truth.columns

In [ ]:

from analysis_village.cc1pi.Constants import CTE as CTE
def TruthInFV(data):
    xmin = -200 + CTE.min_distance_to_wall_x_y
    xmax = 200 - CTE.min_distance_to_wall_x_y
    ymin = -200 + CTE.min_distance_to_wall_x_y
    ymax = 200 - CTE.min_distance_to_wall_x_y
    zmin = CTE.min_distance_to_first_z_wall
    zmax = 500 - CTE.min_distance_to_last_z_wall
    
    pass_fv = (data.x > xmin) & (data.x < xmax) & (data.y > ymin) & (data.y < ymax) & (data.z < zmax) & (data.z > zmin)
    return pass_fv

def IsNu(df):
    is_numu = abs(df.pdg) == 14
    is_nue = abs(df.pdg) == 12
    return is_numu | is_nue   
    
def add_genie_categ_column(df, is_truth_df = False):
    if(is_truth_df):
        truth_df = df
    else:
        truth_df = df.truth # Make a copy to safely assign

    is_inside_fv = TruthInFV(truth_df.position)
    is_nu = IsNu(truth_df)
    is_cc = truth_df.iscc.astype(bool)
    is_nu_mu_cc = is_cc & (abs(truth_df.pdg) == 14)

    genie_categ = pd.Series("other", index=truth_df.index, dtype="object")
    # Apply categories
    genie_categ[~is_nu] = "cosmic"
    genie_categ[is_nu & ~is_inside_fv] = "out_AV_nu"
    genie_categ[is_nu & is_inside_fv & ~is_cc & (abs(truth_df.pdg) == 14)] = "nu_mu_NC" 
    genie_categ[is_nu & is_inside_fv & is_nu_mu_cc & (truth_df.genie_mode == 0)] = "nu_mu_CC_QE" 
    genie_categ[is_nu & is_inside_fv & is_nu_mu_cc & (truth_df.genie_mode == 10)] = "nu_mu_CC_MEC" 
    genie_categ[is_nu & is_inside_fv & is_nu_mu_cc & (truth_df.genie_mode == 1)] = "nu_mu_CC_Res" 
    genie_categ[is_nu & is_inside_fv & is_nu_mu_cc & (truth_df.genie_mode == 2)] = "nu_mu_CC_Dis" 
    
    if(is_truth_df):
        df['genie_categ'] = genie_categ
    else:
        df[('truth', 'genie_categ', '', '','','')] = genie_categ 
    return df

In [ ]:
mc_evt_df = add_genie_categ_column(mc_evt_df, False)
mc_nu_df = add_genie_categ_column(mc_nu_df, False)

In [ ]:
column = ('truth', 'E' ,'','','','')
bins=np.linspace(0, 4, 41)
#np.clip is for including underflow events into the first bin and overflow events into the last bin
# Total MC reco muon momentum: for fake data
eps = 1e-8
var_total_mc = mc_nu_df[column]
#var_total_mc = np.clip(var_total_mc, bins[0], bins[-1] - eps)
weights_total_mc = mc_nu_df.loc[:,  ('slc','wgt','','','','')]

# total generated, for efficiency vector
# Signal event's true muon momentum without event selection
var_truth_signal = mc_nu_df[mc_nu_df.truth.nu_categ == "CC1pi"][column]
#var_truth_signal = np.clip(var_truth_signal, bins[0], bins[-1] - eps)
weight_truth_signal = np.full_like(var_truth_signal, mc_pot_scale, dtype=float)

# --- signal events ---

# Signal event's true muon momentum after the event selection
var_signal_sel_truth = mc_evt_df[mc_evt_df.truth.nu_categ == "CC1pi"][column]
#var_signal_sel_truth = np.clip(var_signal_sel_truth, bins[0], bins[-1] - eps)
weight_true_signal = mc_evt_df.loc[mc_evt_df.truth.nu_categ == "CC1pi",  ('slc','wgt','','','','')]


# Signal event's true muon momentum after the event selection
var_qe = mc_nu_df[mc_nu_df.truth.genie_categ == "nu_mu_CC_QE"][column]
#var_qe = np.clip(var_qe, bins[0], bins[-1] - eps)
weight_qe = mc_nu_df.loc[mc_nu_df.truth.genie_categ == "nu_mu_CC_QE",  ('slc','wgt','','','','')]

# --- Set up figure ---
fig, ax = plt.subplots(figsize=(10, 6))

Normalize = True

#--- Plot normalized histograms ---
#ax.hist(var_total_mc, bins=bins, weights=weights_total_mc, histtype='step',color='black', linewidth=2, label='Total MC', density=Normalize)

ax.hist(var_qe, bins=bins, weights=weight_qe, histtype='step',
        color='red', linewidth=2, label='QE', density=Normalize)

#ax.hist(var_truth_signal, bins=bins, weights=weight_truth_signal, histtype='step', color='blue', linewidth=2, label='Truth Signal (all)', density=Normalize)

ax.hist(var_signal_sel_truth, bins=bins, weights=weight_true_signal, histtype='step',
        color='green', linewidth=2, label='Selected Signal', density=Normalize)

# --- Labels, grid, legend ---
ax.set_xlabel("Neutrino Energy [GeV]")
ax.set_ylabel('Normalized Events')
ax.grid(alpha=0.3)
ax.legend(fontsize=12)

plt.tight_layout()
plt.show()

Draw true (before event selection) and reco (after event selection) muon momentum distributions of signal events.
Print entries for double check.

## Flux

In [ ]:
print(mc_evt_df.truth.columns)

In [ ]:
cov_type = "xsec"
syst_name = "Flux"
n_univ = 100

labels = [var_config.var_labels[1], "Flux Integrated Event Rate"]

save_fig_name = "{}/{}-{}-flux_univ_events".format(save_fig_dir, var_config.var_save_name, syst_name)
ret_flux = get_covariance(cov_type, syst_name, n_univ, 
                          nevts_signal_sel_reco, var_signal_sel_truth, var_signal_sel_reco, var_config.bins, 
                          labels, save_fig=save_fig, save_fig_name=save_fig_name)

In [ ]:
save_fig_name = "{}/{}-{}-flux_covariance".format(save_fig_dir, var_config.var_save_name, syst_name)
plot_heatmap(ret_flux["Covariance"], "Covariance - Flux",
             save_fig=save_fig, save_fig_name=save_fig_name)
save_fig_name = "{}/{}-{}-flux_covariance_frac".format(save_fig_dir, var_config.var_save_name, syst_name)
plot_heatmap(ret_flux["Covariance_Frac"], "Fractional Covariance - Flux",
             save_fig=save_fig, save_fig_name=save_fig_name)
save_fig_name = "{}/{}-{}-flux_correlation".format(save_fig_dir, var_config.var_save_name, syst_name)
plot_heatmap(ret_flux["Correlation"], "Correlation - Flux",
             save_fig=save_fig, save_fig_name=save_fig_name)

## GENIE

In [ ]:
cov_type = "xsec"
syst_name = "GENIE"
n_univ = 100

labels = [var_config.var_labels[1], "Flux Integrated Event Rate"]

save_fig_name = "{}/{}-{}-genie_univ_events".format(save_fig_dir, var_config.var_save_name, syst_name)
ret_genie = get_covariance(cov_type, syst_name, n_univ, 
                          nevts_signal_sel_reco, var_signal_sel_truth, var_signal_sel_reco, var_config.bins, 
                          labels, save_fig=save_fig, save_fig_name=save_fig_name)

In [ ]:
save_fig_name = "{}/{}-{}-genie_covariance".format(save_fig_dir, var_config.var_save_name, syst_name)
plot_heatmap(ret_genie["Covariance"], "Covariance - GENIE",
             save_fig=save_fig, save_fig_name=save_fig_name)
save_fig_name = "{}/{}-{}-genie_covariance_frac".format(save_fig_dir, var_config.var_save_name, syst_name)
plot_heatmap(ret_genie["Covariance_Frac"], "Fractional Covariance - GENIE",
             save_fig=save_fig, save_fig_name=save_fig_name)
save_fig_name = "{}/{}-{}-genie_correlation".format(save_fig_dir, var_config.var_save_name, syst_name)
plot_heatmap(ret_genie["Correlation"], "Correlation - GENIE",
             save_fig=save_fig, save_fig_name=save_fig_name)

## Total

In [ ]:
# add fractional covariance of all systs, then multiply by the CV value to get the covariance
Total_Covariance_Frac = ret_flux["Covariance_Frac"] + ret_genie["Covariance_Frac"] + ret_mcstat["Covariance_Frac"]

Total_Covariance = np.zeros_like(Total_Covariance_Frac)
for i in range(len(var_config.bins)-1):
    for j in range(len(var_config.bins)-1):
        Total_Covariance[i, j] = Total_Covariance_Frac[i, j] * (nevts_signal_sel_reco[i] * nevts_signal_sel_reco[j]) * XSEC_UNIT**2

In [ ]:
ret_mcstat["Covariance"]

In [ ]:
save_fig_name = "{}/{}-{}-total_covariance_frac".format(save_fig_dir, var_config.var_save_name, syst_name)
plot_heatmap(Total_Covariance_Frac, "Total Fractional Covariance",
             save_fig=save_fig, save_fig_name=save_fig_name)

In [ ]:
# fractional uncertainty
frac_uncert_flux = np.sqrt(np.diag(ret_flux["Covariance_Frac"]))
frac_uncert_genie = np.sqrt(np.diag(ret_genie["Covariance_Frac"]))
frac_uncert_mcstat = np.sqrt(np.diag(ret_mcstat["Covariance_Frac"]))
frac_uncert_total = np.sqrt(np.diag(Total_Covariance_Frac))

plt.hist(var_config.bin_centers, bins=var_config.bins, weights=frac_uncert_flux*1e2, histtype="step", color="C0", label="Flux")
plt.hist(var_config.bin_centers, bins=var_config.bins, weights=frac_uncert_genie*1e2, histtype="step", color="C1", label="GENIE")
plt.hist(var_config.bin_centers, bins=var_config.bins, weights=frac_uncert_mcstat*1e2, histtype="step", color="C2", label="MCstat")
plt.hist(var_config.bin_centers, bins=var_config.bins, weights=frac_uncert_total*1e2, histtype="step", color="k", label="Total")

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[0])
plt.ylabel("Uncertainty [%]")
plt.legend()

if save_fig:
    plt.savefig("{}/{}-uncertainty_breakdown.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()


# Singal distribution with error bars from diagonal components of covariance matrix

In [ ]:
# Compute bin centers for error bars
frac_uncert = np.sqrt(np.diag(Total_Covariance_Frac))
plt.errorbar(bin_centers, nevts_signal_sel_reco*XSEC_UNIT, yerr=frac_uncert*nevts_signal_sel_reco*XSEC_UNIT, fmt='o', color='black', label='Subtracted (syst. error)', capsize=3)
plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[0])
plt.ylabel("Events")
plt.legend()

if save_fig:
    plt.savefig("{}/{}-bkg_subtracted_event_rates.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

# Unfolding

## Closure test 
- use MC signal as fake data

In [ ]:
# inspect topology breakdown
plt.figure(figsize=(8, 6))
mc_stack, _, _ = plt.hist(var_per_nu_categ_mc,
                            bins=var_config.bins,
                            weights=weights_per_categ,
                            stacked=True,
                            color=colors,
                            label=mode_labels,
                            edgecolor='none',
                            linewidth=0,
                            density=False,
                            histtype='stepfilled')

totmc, bin_edges = np.histogram(var_total_mc, bins=var_config.bins, weights=weights_total_mc)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# use MC as fake data for closure test
fake_data = totmc
fake_data_err = np.sqrt(totmc)
# plt.step(bin_edges[:-1], fake_data, where='post', label="Data")  # line
plt.errorbar(bin_centers, fake_data, yerr=fake_data_err, fmt='o', color='black')  # error bars

accum_sum = [np.sum(data) for data in mc_stack]
accum_sum = [0.] + accum_sum
total_sum = accum_sum[-1]
individual_sums = [accum_sum[i + 1] - accum_sum[i] for i in range(len(accum_sum) - 1)]
fractions = [(count / total_sum) * 100 for count in individual_sums]
legend_labels = [f"{label} ({frac:.1f}%)" for label, frac in zip(mode_labels[::-1], fractions[::-1])]
legend_labels.append("Fake Data")
plt.legend(legend_labels, loc='upper left', fontsize=10, frameon=False, ncol=3, bbox_to_anchor=(0.05, 0.98))

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[1])
plt.ylim(0., 1.3 * fake_data.max())
plt.ylabel("Events")

if save_fig:
    plt.savefig("{}/{}-topology_breakdown.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
# inspect GENIE mode breakdown
plt.figure(figsize=(8, 6))
mc_stack, _, _ = plt.hist(var_per_genie_mode_mc,
                            bins=var_config.bins,
                            weights=weights_per_genie_mode,
                            stacked=True,
                            color=genie_mode_colors,
                            label=genie_mode_labels,
                            edgecolor='none',
                            linewidth=0,
                            density=False,
                            histtype='stepfilled')

totmc, bin_edges = np.histogram(var_total_mc, bins=var_config.bins, weights=weights_total_mc)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# use MC as fake data for closure test
fake_data = totmc
fake_data_err = np.sqrt(totmc)
# plt.step(bin_edges[:-1], fake_data, where='post', label="Data")  # line
plt.errorbar(bin_centers, fake_data, yerr=fake_data_err, fmt='o', color='black')  # error bars

accum_sum = [np.sum(data) for data in mc_stack]
accum_sum = [0.] + accum_sum
total_sum = accum_sum[-1]
individual_sums = [accum_sum[i + 1] - accum_sum[i] for i in range(len(accum_sum) - 1)]
fractions = [(count / total_sum) * 100 for count in individual_sums]
legend_labels = [f"{label} ({frac:.1f}%)" for label, frac in zip(genie_mode_labels[::-1], fractions[::-1])]
legend_labels.append("Fake Data")
plt.legend(legend_labels, loc='upper left', fontsize=10, frameon=False, ncol=3, bbox_to_anchor=(0.05, 0.98))

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[1])
plt.ylim(0., 1.3 * fake_data.max())
plt.ylabel("Events")

if save_fig:
    plt.savefig("{}/{}-genie_mode_breakdown.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
# closure test -- just use MC stat uncertainty
C_type = 2
Norm_type = 0.5
Measured = nevts_signal_sel_reco * XSEC_UNIT # = fake_data - fake_background
Model = nevts_signal_truth * XSEC_UNIT
Covariance = ret_mcstat["Covariance"] * XSEC_UNIT**2
# Covariance = ret_flux["Covariance"]
unfold = WienerSVD(Response, Model, Measured, Covariance, C_type, Norm_type)

In [ ]:
unfold['unfold']

In [ ]:
unfold['UnfoldCov']

In [ ]:
def chi2(data, model, cov):
    return (data - model) @ np.linalg.inv(cov) @ (data - model)

In [ ]:
unfold['unfold']

In [ ]:
unfold['AddSmear'] @ nevts_signal_truth

In [ ]:
Unfold = unfold['unfold']
UnfoldCov = unfold['UnfoldCov']
Unfold_uncert = np.sqrt(np.diag(UnfoldCov))

step_handle, = plt.step(bin_edges, np.append(Unfold, Unfold[-1]), where='post', label='Unfolded', color='black')
bar_handle = plt.bar(
    bin_centers,
    2*Unfold_uncert,
    width=(bin_edges[1] - bin_edges[0]) * 1.,
    bottom=Unfold - Unfold_uncert,
    color='gray',
    alpha=0.5,
    linewidth=0,
    label='Unfolded Stat. error (box)'
)
reco_handle, = plt.plot(bin_centers, Measured, 'o', label='Reco. signal')
Model_smear = unfold['AddSmear'] @ Model
true_handle, = plt.plot(bin_centers, Model_smear, 'o', label='$A_c \\times$ True signal')

chi2_val = chi2(Unfold, Model_smear, UnfoldCov)
plt.text(0.95, 0.50, f"$ \\chi^2 = $ {chi2_val:.2f}", ha='right', va='center', 
         transform=plt.gca().transAxes, fontsize=12)

# Custom order for legend
handles = [step_handle, bar_handle, reco_handle, true_handle]
labels = [
    'Unfolded',
    'Unfolded error',
    'Reco. signal',
    '$A_c \\times$True signal'
]

plt.legend(handles, labels)
plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[0])
# plt.ylim(0., 5000.)
plt.ylabel("Events")

if save_fig:
    plt.savefig("{}/{}-unfolded_event_rates.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
save_fig_name = "{}/{}-{}-add_smear".format(save_fig_dir, var_config.var_save_name, syst_name)
plot_labels = [var_config.var_labels[2], var_config.var_labels[1]]
plot_heatmap(unfold["AddSmear"], "$A_c$", plot_labels=plot_labels,
             save_fig=save_fig, save_fig_name=save_fig_name)

## Fake Data Tests

- use alternate MC as fake data

### MEC scale

In [ ]:
mec_scale = 0.5
weights_fake_data = np.ones(len(var_total_mc))
weights_fake_data[mc_evt_df.truth.genie_mode == 10] = mec_scale

In [ ]:
plt.figure(figsize=(8, 6))
mc_stack, _, _ = plt.hist(var_per_nu_categ_mc,
                            bins=var_config.bins,
                            weights=weights_per_categ,
                            stacked=True,
                            color=colors,
                            label=mode_labels,
                            edgecolor='none',
                            linewidth=0,
                            density=False,
                            histtype='stepfilled')
totmc, bin_edges = np.histogram(var_total_mc, bins=var_config.bins, weights=weights_total_mc * weights_fake_data)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# use MC as fake data for closure test
fake_data = totmc
fake_data_err = np.sqrt(totmc)
plt.errorbar(bin_centers, fake_data, yerr=fake_data_err, fmt='o', color='black')  # error bars

background_cv = mc_stack[-1] - mc_stack[0]
# plt.hist(bin_centers, weights=fake_data - background_cv, bins=var_config.bins, histtype="step", color="k", label="Background")

accum_sum = [np.sum(data) for data in mc_stack]
accum_sum = [0.] + accum_sum
total_sum = accum_sum[-1]
individual_sums = [accum_sum[i + 1] - accum_sum[i] for i in range(len(accum_sum) - 1)]
fractions = [(count / total_sum) * 100 for count in individual_sums]
legend_labels = [f"{label} ({frac:.1f}%)" for label, frac in zip(mode_labels[::-1], fractions[::-1])]
legend_labels.append("Fake Data (MEC x{})".format(mec_scale))
plt.legend(legend_labels, loc='upper left', fontsize=10, frameon=False, ncol=3, bbox_to_anchor=(0.05, 0.98))

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[1])
plt.ylim(0., 1.3 * fake_data.max())
plt.ylabel("Events")

if save_fig:
    plt.savefig("{}/{}-topology_breakdown.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
# inspect GENIE mode breakdown
plt.figure(figsize=(8, 6))
mc_stack, _, _ = plt.hist(var_per_genie_mode_mc,
                            bins=var_config.bins,
                            weights=weights_per_genie_mode,
                            stacked=True,
                            color=genie_mode_colors,
                            label=genie_mode_labels,
                            edgecolor='none',
                            linewidth=0,
                            density=False,
                            histtype='stepfilled')

totmc, bin_edges = np.histogram(var_total_mc, bins=var_config.bins, weights=weights_total_mc * weights_fake_data)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# use MC as fake data for closure test
fake_data = totmc
fake_data_err = np.sqrt(totmc)
# plt.step(bin_edges[:-1], fake_data, where='post', label="Data")  # line
plt.errorbar(bin_centers, fake_data, yerr=fake_data_err, fmt='o', color='black')  # error bars

accum_sum = [np.sum(data) for data in mc_stack]
accum_sum = [0.] + accum_sum
total_sum = accum_sum[-1]
individual_sums = [accum_sum[i + 1] - accum_sum[i] for i in range(len(accum_sum) - 1)]
fractions = [(count / total_sum) * 100 for count in individual_sums]
legend_labels = [f"{label} ({frac:.1f}%)" for label, frac in zip(genie_mode_labels[::-1], fractions[::-1])]
legend_labels.append("Fake Data (MEC x{})".format(mec_scale))
plt.legend(legend_labels, loc='upper left', fontsize=10, frameon=False, ncol=3, bbox_to_anchor=(0.05, 0.98))

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[1])
plt.ylim(0., 1.3 * fake_data.max())
plt.ylabel("Events")

if save_fig:
    plt.savefig("{}/{}-genie_mode_breakdown.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
# fake data test -- MC stat + xsec cov
C_type = 2
Norm_type = 0.5
Measured = fake_data * XSEC_UNIT - background_cv * XSEC_UNIT
Model = nevts_signal_truth * XSEC_UNIT

# use MC stat and xsec cov
Covariance_Frac = ret_genie["Covariance_Frac"] + ret_mcstat["Covariance_Frac"]

Covariance = np.zeros_like(Covariance_Frac)
for i in range(len(var_config.bins)-1):
    for j in range(len(var_config.bins)-1):
        Covariance[i, j] = Covariance_Frac[i, j] * (nevts_signal_sel_reco[i] * nevts_signal_sel_reco[j]) * XSEC_UNIT**2

unfold = WienerSVD(Response, Model, Measured, Covariance, C_type, Norm_type)

In [ ]:
# true cross section for alt MC used asfake data
var_fakedata_signal_truth = mc_nu_df[mc_nu_df.truth.nu_categ == "CC1pi"][var_config.var_nu_col]
var_fakedata_signal_truth = np.clip(var_fakedata_signal_truth, var_config.bins[0], var_config.bins[-1] - eps)
weight_fakedata_signal_truth = np.ones(len(var_fakedata_signal_truth))
nevts_fakedata_signal_truth, _, _ = plt.hist(var_fakedata_signal_truth, bins=var_config.bins, weights=weight_fakedata_signal_truth, histtype="step", label="True Signal")
weight_fakedata_signal_truth[mc_nu_df[mc_nu_df.truth.nu_categ == "CC1pi"].truth.genie_mode == 10] = mec_scale
nevts_fakedata_signal_truth, _, _ = plt.hist(var_fakedata_signal_truth, bins=var_config.bins, weights=weight_fakedata_signal_truth, histtype="step", label="MEC x2 True Signal")
plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
Unfold = unfold['unfold']
UnfoldCov = unfold['UnfoldCov']
Unfold_uncert = np.sqrt(np.diag(UnfoldCov))

step_handle, = plt.step(bin_edges, np.append(Unfold, Unfold[-1]), where='post', label='Unfolded', color='black')
bar_handle = plt.bar(
    bin_centers,
    2*Unfold_uncert,
    width=(bin_edges[1] - bin_edges[0]) * 1.,
    bottom=Unfold - Unfold_uncert,
    color='gray',
    alpha=0.5,
    linewidth=0,
    label='Unfolded Stat. error (box)'
)
reco_handle, = plt.plot(bin_centers, Measured, 'o', label='Reco. signal')
Model_smear = unfold['AddSmear'] @ Model
true_handle, = plt.plot(bin_centers, Model_smear, 'o', label='$A_c \\times$ True signal')
Fakedata_Model_smear = unfold['AddSmear'] @ nevts_fakedata_signal_truth*XSEC_UNIT
fake_true_handle, = plt.plot(bin_centers, Fakedata_Model_smear, 'o', label='Fake Data')

chi2_val = chi2(Unfold, Model_smear, UnfoldCov)
chi2_val_fakedata = chi2(Unfold, Fakedata_Model_smear, UnfoldCov)
plt.text(0.95, 0.50, f"$ \\chi^2 = $ {chi2_val:.2f}", ha='right', va='center', 
          transform=plt.gca().transAxes, fontsize=12)

# Custom order for legend
handles = [step_handle, bar_handle, reco_handle, true_handle, fake_true_handle]
labels = [
    'Unfolded',
    'Unfolded error',
    'Reco. signal',
    '$A_c \\times$Nominal Truth ($\\chi^2 = {:.2f}/{}$)'.format(chi2_val, len(var_config.bins)-1),
    '$A_c \\times$ MECx{} Truth ($\\chi^2 = {:.2f}/{}$)'.format(mec_scale, chi2_val_fakedata, len(var_config.bins)-1),
]

plt.legend(handles, labels, fontsize=12)
plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[0])
# plt.ylim(0., 5000.)
plt.ylabel("Events")

if save_fig:
    plt.savefig("{}/{}-unfolded_event_rates.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
save_fig_name = "{}/{}-{}-add_smear".format(save_fig_dir, var_config.var_save_name, syst_name)
plot_labels = [var_config.var_labels[2], var_config.var_labels[1]]
plot_heatmap(unfold["AddSmear"], "$A_c$", plot_labels=plot_labels,
             save_fig=save_fig, save_fig_name=save_fig_name)

### QE scale

In [ ]:
qe_scale = 1.2
weights_fake_data = np.ones(len(var_total_mc))
weights_fake_data[mc_evt_df.truth.genie_mode == 0] = qe_scale

In [ ]:
plt.figure(figsize=(8, 6))
mc_stack, _, _ = plt.hist(var_per_nu_categ_mc,
                            bins=var_config.bins,
                            weights=weights_per_categ,
                            stacked=True,
                            color=colors,
                            label=mode_labels,
                            edgecolor='none',
                            linewidth=0,
                            density=False,
                            histtype='stepfilled')
totmc, bin_edges = np.histogram(var_total_mc, bins=var_config.bins, weights=weights_total_mc * weights_fake_data)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# use MC as fake data for closure test
fake_data = totmc
fake_data_err = np.sqrt(totmc)
plt.errorbar(bin_centers, fake_data, yerr=fake_data_err, fmt='o', color='black')  # error bars

background_cv = mc_stack[-1] - mc_stack[0]
# plt.hist(bin_centers, weights=fake_data - background_cv, bins=var_config.bins, histtype="step", color="k", label="Background")

accum_sum = [np.sum(data) for data in mc_stack]
accum_sum = [0.] + accum_sum
total_sum = accum_sum[-1]
individual_sums = [accum_sum[i + 1] - accum_sum[i] for i in range(len(accum_sum) - 1)]
fractions = [(count / total_sum) * 100 for count in individual_sums]
legend_labels = [f"{label} ({frac:.1f}%)" for label, frac in zip(mode_labels[::-1], fractions[::-1])]
legend_labels.append("Fake Data (QE x{})".format(qe_scale))
plt.legend(legend_labels, loc='upper left', fontsize=10, frameon=False, ncol=3, bbox_to_anchor=(0.05, 0.98))

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[1])
plt.ylim(0., 1.3 * fake_data.max())
plt.ylabel("Events")

if save_fig:
    plt.savefig("{}/{}-topology_breakdown.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
# inspect GENIE mode breakdown
plt.figure(figsize=(8, 6))
mc_stack, _, _ = plt.hist(var_per_genie_mode_mc,
                            bins=var_config.bins,
                            weights=weights_per_genie_mode,
                            stacked=True,
                            color=genie_mode_colors,
                            label=genie_mode_labels,
                            edgecolor='none',
                            linewidth=0,
                            density=False,
                            histtype='stepfilled')

totmc, bin_edges = np.histogram(var_total_mc, bins=var_config.bins, weights=weights_total_mc * weights_fake_data)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# use MC as fake data for closure test
fake_data = totmc
fake_data_err = np.sqrt(totmc)
# plt.step(bin_edges[:-1], fake_data, where='post', label="Data")  # line
plt.errorbar(bin_centers, fake_data, yerr=fake_data_err, fmt='o', color='black')  # error bars

accum_sum = [np.sum(data) for data in mc_stack]
accum_sum = [0.] + accum_sum
total_sum = accum_sum[-1]
individual_sums = [accum_sum[i + 1] - accum_sum[i] for i in range(len(accum_sum) - 1)]
fractions = [(count / total_sum) * 100 for count in individual_sums]
legend_labels = [f"{label} ({frac:.1f}%)" for label, frac in zip(genie_mode_labels[::-1], fractions[::-1])]
legend_labels.append("Fake Data (QE x{})".format(qe_scale))
plt.legend(legend_labels, loc='upper left', fontsize=10, frameon=False, ncol=3, bbox_to_anchor=(0.05, 0.98))

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[1])
plt.ylim(0., 1.3 * fake_data.max())
plt.ylabel("Events")

if save_fig:
    plt.savefig("{}/{}-genie_mode_breakdown.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
# fake data test -- MC stat + xsec cov
C_type = 2
Norm_type = 0.5
Measured = fake_data * XSEC_UNIT - background_cv * XSEC_UNIT
Model = nevts_signal_truth * XSEC_UNIT

# use MC stat and xsec cov
Covariance_Frac = ret_genie["Covariance_Frac"] + ret_mcstat["Covariance_Frac"]

Covariance = np.zeros_like(Covariance_Frac)
for i in range(len(var_config.bins)-1):
    for j in range(len(var_config.bins)-1):
        Covariance[i, j] = Covariance_Frac[i, j] * (nevts_signal_sel_reco[i] * nevts_signal_sel_reco[j]) * XSEC_UNIT**2

unfold = WienerSVD(Response, Model, Measured, Covariance, C_type, Norm_type)

In [ ]:
# true cross section for alt MC used asfake data
var_fakedata_signal_truth = mc_nu_df[mc_nu_df.truth.nu_categ == "CC1pi"][var_config.var_nu_col]
var_fakedata_signal_truth = np.clip(var_fakedata_signal_truth, var_config.bins[0], var_config.bins[-1] - eps)
weight_fakedata_signal_truth = np.ones(len(var_fakedata_signal_truth))
nevts_fakedata_signal_truth, _, _ = plt.hist(var_fakedata_signal_truth, bins=var_config.bins, weights=weight_fakedata_signal_truth, histtype="step", label="True Signal")
weight_fakedata_signal_truth[mc_nu_df[mc_nu_df.truth.nu_categ == "CC1pi"].truth.genie_mode == 0] = qe_scale
nevts_fakedata_signal_truth, _, _ = plt.hist(var_fakedata_signal_truth, bins=var_config.bins, weights=weight_fakedata_signal_truth, histtype="step", label="QE x{} True Signal".format(qe_scale))
plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
Unfold = unfold['unfold']
UnfoldCov = unfold['UnfoldCov']
Unfold_uncert = np.sqrt(np.diag(UnfoldCov))

step_handle, = plt.step(bin_edges, np.append(Unfold, Unfold[-1]), where='post', label='Unfolded', color='black')
bar_handle = plt.bar(
    bin_centers,
    2*Unfold_uncert,
    width=(bin_edges[1] - bin_edges[0]) * 1.,
    bottom=Unfold - Unfold_uncert,
    color='gray',
    alpha=0.5,
    linewidth=0,
    label='Unfolded Stat. error (box)'
)
reco_handle, = plt.plot(bin_centers, Measured, 'o', label='Reco. signal')
Model_smear = unfold['AddSmear'] @ Model
true_handle, = plt.plot(bin_centers, Model_smear, 'o', label='$A_c \\times$ True signal')
Fakedata_Model_smear = unfold['AddSmear'] @ nevts_fakedata_signal_truth*XSEC_UNIT
fake_true_handle, = plt.plot(bin_centers, Fakedata_Model_smear, 'o', label='Fake Data')

chi2_val = chi2(Unfold, Model_smear, UnfoldCov)
chi2_val_fakedata = chi2(Unfold, Fakedata_Model_smear, UnfoldCov)
plt.text(0.95, 0.50, f"$ \\chi^2 = $ {chi2_val:.2f}", ha='right', va='center', 
          transform=plt.gca().transAxes, fontsize=12)

# Custom order for legend
handles = [step_handle, bar_handle, reco_handle, true_handle, fake_true_handle]
labels = [
    'Unfolded',
    'Unfolded error',
    'Reco. signal',
    '$A_c \\times$Nominal Truth ($\\chi^2 = {:.2f}/{}$)'.format(chi2_val, len(var_config.bins)-1),
    '$A_c \\times$ QE x{} Truth ($\\chi^2 = {:.2f}/{}$)'.format(qe_scale, chi2_val_fakedata, len(var_config.bins)-1),
]

plt.legend(handles, labels, fontsize=12)
plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[0])
# plt.ylim(0., 5000.)
plt.ylabel("Events")

if save_fig:
    plt.savefig("{}/{}-unfolded_event_rates.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
save_fig_name = "{}/{}-{}-add_smear".format(save_fig_dir, var_config.var_save_name, syst_name)
plot_labels = [var_config.var_labels[2], var_config.var_labels[1]]
plot_heatmap(unfold["AddSmear"], "$A_c$", plot_labels=plot_labels,
             save_fig=save_fig, save_fig_name=save_fig_name)

# Resonant scale

In [ ]:
resonant_scale = 1.2
weights_fake_data = np.ones(len(var_total_mc))
weights_fake_data[mc_evt_df.truth.genie_mode == 1] = resonant_scale

In [ ]:
plt.figure(figsize=(8, 6))
mc_stack, _, _ = plt.hist(var_per_nu_categ_mc,
                            bins=var_config.bins,
                            weights=weights_per_categ,
                            stacked=True,
                            color=colors,
                            label=mode_labels,
                            edgecolor='none',
                            linewidth=0,
                            density=False,
                            histtype='stepfilled')
totmc, bin_edges = np.histogram(var_total_mc, bins=var_config.bins, weights=weights_total_mc * weights_fake_data)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# use MC as fake data for closure test
fake_data = totmc
fake_data_err = np.sqrt(totmc)
plt.errorbar(bin_centers, fake_data, yerr=fake_data_err, fmt='o', color='black')  # error bars

background_cv = mc_stack[-1] - mc_stack[0]
# plt.hist(bin_centers, weights=fake_data - background_cv, bins=var_config.bins, histtype="step", color="k", label="Background")

accum_sum = [np.sum(data) for data in mc_stack]
accum_sum = [0.] + accum_sum
total_sum = accum_sum[-1]
individual_sums = [accum_sum[i + 1] - accum_sum[i] for i in range(len(accum_sum) - 1)]
fractions = [(count / total_sum) * 100 for count in individual_sums]
legend_labels = [f"{label} ({frac:.1f}%)" for label, frac in zip(mode_labels[::-1], fractions[::-1])]
legend_labels.append("Fake Data (QE x{})".format(resonant_scale))
plt.legend(legend_labels, loc='upper left', fontsize=10, frameon=False, ncol=3, bbox_to_anchor=(0.05, 0.98))

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[1])
plt.ylim(0., 1.3 * fake_data.max())
plt.ylabel("Events")

if save_fig:
    plt.savefig("{}/{}-topology_breakdown.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
# inspect GENIE mode breakdown
plt.figure(figsize=(8, 6))
mc_stack, _, _ = plt.hist(var_per_genie_mode_mc,
                            bins=var_config.bins,
                            weights=weights_per_genie_mode,
                            stacked=True,
                            color=genie_mode_colors,
                            label=genie_mode_labels,
                            edgecolor='none',
                            linewidth=0,
                            density=False,
                            histtype='stepfilled')

totmc, bin_edges = np.histogram(var_total_mc, bins=var_config.bins, weights=weights_total_mc * weights_fake_data)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# use MC as fake data for closure test
fake_data = totmc
fake_data_err = np.sqrt(totmc)
# plt.step(bin_edges[:-1], fake_data, where='post', label="Data")  # line
plt.errorbar(bin_centers, fake_data, yerr=fake_data_err, fmt='o', color='black')  # error bars

accum_sum = [np.sum(data) for data in mc_stack]
accum_sum = [0.] + accum_sum
total_sum = accum_sum[-1]
individual_sums = [accum_sum[i + 1] - accum_sum[i] for i in range(len(accum_sum) - 1)]
fractions = [(count / total_sum) * 100 for count in individual_sums]
legend_labels = [f"{label} ({frac:.1f}%)" for label, frac in zip(genie_mode_labels[::-1], fractions[::-1])]
legend_labels.append("Fake Data (Resonant x{})".format(resonant_scale))
plt.legend(legend_labels, loc='upper left', fontsize=10, frameon=False, ncol=3, bbox_to_anchor=(0.05, 0.98))

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[1])
plt.ylim(0., 1.3 * fake_data.max())
plt.ylabel("Events")

if save_fig:
    plt.savefig("{}/{}-genie_mode_breakdown.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
# fake data test -- MC stat + xsec cov
C_type = 2
Norm_type = 0.5
Measured = fake_data * XSEC_UNIT - background_cv * XSEC_UNIT
Model = nevts_signal_truth * XSEC_UNIT

# use MC stat and xsec cov
Covariance_Frac = ret_genie["Covariance_Frac"] + ret_mcstat["Covariance_Frac"]

Covariance = np.zeros_like(Covariance_Frac)
for i in range(len(var_config.bins)-1):
    for j in range(len(var_config.bins)-1):
        Covariance[i, j] = Covariance_Frac[i, j] * (nevts_signal_sel_reco[i] * nevts_signal_sel_reco[j]) * XSEC_UNIT**2
print(Covariance)
unfold = WienerSVD(Response, Model, Measured, Covariance, C_type, Norm_type)

In [ ]:
# true cross section for alt MC used asfake data
var_fakedata_signal_truth = mc_nu_df[mc_nu_df.truth.nu_categ == "CC1pi"][var_config.var_nu_col]
var_fakedata_signal_truth = np.clip(var_fakedata_signal_truth, var_config.bins[0], var_config.bins[-1] - eps)
weight_fakedata_signal_truth = np.ones(len(var_fakedata_signal_truth))
nevts_fakedata_signal_truth, _, _ = plt.hist(var_fakedata_signal_truth, bins=var_config.bins, weights=weight_fakedata_signal_truth*weight_truth_signal, histtype="step", label="True Signal")
weight_fakedata_signal_truth[mc_nu_df[mc_nu_df.truth.nu_categ == "CC1pi"].truth.genie_mode == 1] = resonant_scale
nevts_fakedata_signal_truth, _, _ = plt.hist(var_fakedata_signal_truth, bins=var_config.bins, weights=weight_fakedata_signal_truth*weight_truth_signal, histtype="step", label="Resonant x{} True Signal".format(qe_scale))
plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
Unfold = unfold['unfold']
UnfoldCov = unfold['UnfoldCov']
Unfold_uncert = np.sqrt(np.diag(UnfoldCov))

step_handle, = plt.step(bin_edges, np.append(Unfold, Unfold[-1]), where='post', label='Unfolded', color='black')
bar_handle = plt.bar(
    bin_centers,
    2*Unfold_uncert,
    width=(bin_edges[1] - bin_edges[0]) * 1.,
    bottom=Unfold - Unfold_uncert,
    color='gray',
    alpha=0.5,
    linewidth=0,
    label='Unfolded Stat. error (box)'
)
reco_handle, = plt.plot(bin_centers, Measured, 'o', label='Reco. signal')
Model_smear = unfold['AddSmear'] @ Model
true_handle, = plt.plot(bin_centers, Model_smear, 'o', label='$A_c \\times$ True signal')
Fakedata_Model_smear = unfold['AddSmear'] @ nevts_fakedata_signal_truth*XSEC_UNIT
fake_true_handle, = plt.plot(bin_centers, Fakedata_Model_smear, 'o', label='Fake Data')

chi2_val = chi2(Unfold, Model_smear, UnfoldCov)
chi2_val_fakedata = chi2(Unfold, Fakedata_Model_smear, UnfoldCov)
# plt.text(0.95, 0.50, f"$ \\chi^2 = $ {chi2_val:.2f}", ha='right', va='center', 
#          transform=plt.gca().transAxes, fontsize=12)

# Custom order for legend
handles = [step_handle, bar_handle, reco_handle, true_handle, fake_true_handle]
labels = [
    'Unfolded',
    'Unfolded error',
    'Reco. signal',
    '$A_c \\times$Nominal Truth ($\\chi^2 = {:.2f}/{}$)'.format(chi2_val, len(var_config.bins)-1),
    '$A_c \\times$ Resonant x{} Truth ($\\chi^2 = {:.2f}/{}$)'.format(resonant_scale, chi2_val_fakedata, len(var_config.bins)-1),
]

plt.legend(handles, labels, fontsize=12)
plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[0])
# plt.ylim(0., 5000.)
plt.ylabel("Events")

if save_fig:
    plt.savefig("{}/{}-unfolded_event_rates.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
save_fig_name = "{}/{}-{}-add_smear".format(save_fig_dir, var_config.var_save_name, syst_name)
plot_labels = [var_config.var_labels[2], var_config.var_labels[1]]
plot_heatmap(unfold["AddSmear"], "$A_c$", plot_labels=plot_labels,
             save_fig=save_fig, save_fig_name=save_fig_name)

# 1p background scale

In [ ]:
Np_scale = 2
weights_fake_data = np.ones(len(var_total_mc))
weights_fake_data[mc_evt_df.truth.nu_categ == "CC_mu_0pi_1p"] = Np_scale

In [ ]:
plt.figure(figsize=(8, 6))
mc_stack, _, _ = plt.hist(var_per_nu_categ_mc,
                            bins=var_config.bins,
                            weights=weights_per_categ,
                            stacked=True,
                            color=colors,
                            label=mode_labels,
                            edgecolor='none',
                            linewidth=0,
                            density=False,
                            histtype='stepfilled')
totmc, bin_edges = np.histogram(var_total_mc, bins=var_config.bins, weights=weights_total_mc * weights_fake_data)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# use MC as fake data for closure test
fake_data = totmc
fake_data_err = np.sqrt(totmc)
plt.errorbar(bin_centers, fake_data, yerr=fake_data_err, fmt='o', color='black')  # error bars

background_cv = mc_stack[-1] - mc_stack[0]
# plt.hist(bin_centers, weights=fake_data - background_cv, bins=var_config.bins, histtype="step", color="k", label="Background")

accum_sum = [np.sum(data) for data in mc_stack]
accum_sum = [0.] + accum_sum
total_sum = accum_sum[-1]
individual_sums = [accum_sum[i + 1] - accum_sum[i] for i in range(len(accum_sum) - 1)]
fractions = [(count / total_sum) * 100 for count in individual_sums]
legend_labels = [f"{label} ({frac:.1f}%)" for label, frac in zip(mode_labels[::-1], fractions[::-1])]
legend_labels.append("Fake Data (1p x{})".format(Np_scale))
plt.legend(legend_labels, loc='upper left', fontsize=10, frameon=False, ncol=3, bbox_to_anchor=(0.05, 0.98))

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[1])
plt.ylim(0., 1.3 * fake_data.max())
plt.ylabel("Events")

if save_fig:
    plt.savefig("{}/{}-topology_breakdown.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
# inspect GENIE mode breakdown
plt.figure(figsize=(8, 6))
mc_stack, _, _ = plt.hist(var_per_genie_mode_mc,
                            bins=var_config.bins,
                            weights=weights_per_genie_mode,
                            stacked=True,
                            color=genie_mode_colors,
                            label=genie_mode_labels,
                            edgecolor='none',
                            linewidth=0,
                            density=False,
                            histtype='stepfilled')

totmc, bin_edges = np.histogram(var_total_mc, bins=var_config.bins, weights=weights_total_mc * weights_fake_data)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# use MC as fake data for closure test
fake_data = totmc
fake_data_err = np.sqrt(totmc)
# plt.step(bin_edges[:-1], fake_data, where='post', label="Data")  # line
plt.errorbar(bin_centers, fake_data, yerr=fake_data_err, fmt='o', color='black')  # error bars

accum_sum = [np.sum(data) for data in mc_stack]
accum_sum = [0.] + accum_sum
total_sum = accum_sum[-1]
individual_sums = [accum_sum[i + 1] - accum_sum[i] for i in range(len(accum_sum) - 1)]
fractions = [(count / total_sum) * 100 for count in individual_sums]
legend_labels = [f"{label} ({frac:.1f}%)" for label, frac in zip(genie_mode_labels[::-1], fractions[::-1])]
legend_labels.append("Fake Data (1p x{})".format(Np_scale))
plt.legend(legend_labels, loc='upper left', fontsize=10, frameon=False, ncol=3, bbox_to_anchor=(0.05, 0.98))

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[1])
plt.ylim(0., 1.3 * fake_data.max())
plt.ylabel("Events")

if save_fig:
    plt.savefig("{}/{}-genie_mode_breakdown.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
# fake data test -- MC stat + xsec cov
C_type = 2
Norm_type = 0.5
Measured = fake_data * XSEC_UNIT - background_cv * XSEC_UNIT
Model = nevts_signal_truth * XSEC_UNIT

# use MC stat and xsec cov
Covariance_Frac = ret_genie["Covariance_Frac"] + ret_mcstat["Covariance_Frac"]

Covariance = np.zeros_like(Covariance_Frac)
for i in range(len(var_config.bins)-1):
    for j in range(len(var_config.bins)-1):
        Covariance[i, j] = Covariance_Frac[i, j] * (nevts_signal_sel_reco[i] * nevts_signal_sel_reco[j]) * XSEC_UNIT**2

unfold = WienerSVD(Response, Model, Measured, Covariance, C_type, Norm_type)

In [ ]:
# true cross section for alt MC used asfake data
var_fakedata_signal_truth = mc_nu_df[mc_nu_df.truth.nu_categ == "CC1pi"][var_config.var_nu_col]
var_fakedata_signal_truth = np.clip(var_fakedata_signal_truth, var_config.bins[0], var_config.bins[-1] - eps)
weight_fakedata_signal_truth = np.ones(len(var_fakedata_signal_truth))
nevts_fakedata_signal_truth, _, _ = plt.hist(var_fakedata_signal_truth, bins=var_config.bins, weights=weight_fakedata_signal_truth, histtype="step", label="True Signal")
weight_fakedata_signal_truth[mc_nu_df[mc_nu_df.truth.nu_categ == "CC1pi"].truth.nu_categ == "CC_mu_0pi_1p"] = Np_scale
nevts_fakedata_signal_truth, _, _ = plt.hist(var_fakedata_signal_truth, bins=var_config.bins, weights=weight_fakedata_signal_truth, histtype="step", label="Np x{} True Signal".format(Np_scale))
plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
Unfold = unfold['unfold']
UnfoldCov = unfold['UnfoldCov']
Unfold_uncert = np.sqrt(np.diag(UnfoldCov))

step_handle, = plt.step(bin_edges, np.append(Unfold, Unfold[-1]), where='post', label='Unfolded', color='black')
bar_handle = plt.bar(
    bin_centers,
    2*Unfold_uncert,
    width=(bin_edges[1] - bin_edges[0]) * 1.,
    bottom=Unfold - Unfold_uncert,
    color='gray',
    alpha=0.5,
    linewidth=0,
    label='Unfolded Stat. error (box)'
)
reco_handle, = plt.plot(bin_centers, Measured, 'o', label='Reco. signal')
Model_smear = unfold['AddSmear'] @ Model
true_handle, = plt.plot(bin_centers, Model_smear, 'o', label='$A_c \\times$ True signal')
Fakedata_Model_smear = unfold['AddSmear'] @ nevts_fakedata_signal_truth*XSEC_UNIT
fake_true_handle, = plt.plot(bin_centers, Fakedata_Model_smear, 'o', label='Fake Data')

chi2_val = chi2(Unfold, Model_smear, UnfoldCov)
chi2_val_fakedata = chi2(Unfold, Fakedata_Model_smear, UnfoldCov)
# plt.text(0.95, 0.50, f"$ \\chi^2 = $ {chi2_val:.2f}", ha='right', va='center', 
#          transform=plt.gca().transAxes, fontsize=12)

# Custom order for legend
handles = [step_handle, bar_handle, reco_handle, true_handle, fake_true_handle]
labels = [
    'Unfolded',
    'Unfolded error',
    'Reco. signal',
    '$A_c \\times$Nominal Truth ($\\chi^2 = {:.2f}/{}$)'.format(chi2_val, len(var_config.bins)-1),
    '$A_c \\times$ 1p x{} Truth ($\\chi^2 = {:.2f}/{}$)'.format(Np_scale, chi2_val_fakedata, len(var_config.bins)-1),
]

plt.legend(handles, labels, fontsize=12)
plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[0])
# plt.ylim(0., 5000.)
plt.ylabel("Events")

if save_fig:
    plt.savefig("{}/{}-unfolded_event_rates.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
save_fig_name = "{}/{}-{}-add_smear".format(save_fig_dir, var_config.var_save_name, syst_name)
plot_labels = [var_config.var_labels[2], var_config.var_labels[1]]
plot_heatmap(unfold["AddSmear"], "$A_c$", plot_labels=plot_labels,
             save_fig=save_fig, save_fig_name=save_fig_name)

### Signal scale

In [ ]:
sig_scale = 1.2
weights_fake_data = np.ones(len(var_total_mc))
weights_fake_data[mc_evt_df.truth.nu_categ == "CC1pi"] = sig_scale

In [ ]:
plt.figure(figsize=(8, 6))
mc_stack, _, _ = plt.hist(var_per_nu_categ_mc,
                            bins=var_config.bins,
                            weights=weights_per_categ,
                            stacked=True,
                            color=colors,
                            label=mode_labels,
                            edgecolor='none',
                            linewidth=0,
                            density=False,
                            histtype='stepfilled')
totmc, bin_edges = np.histogram(var_total_mc, bins=var_config.bins, weights=weights_total_mc * weights_fake_data)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# use MC as fake data for closure test
fake_data = totmc
fake_data_err = np.sqrt(totmc)
plt.errorbar(bin_centers, fake_data, yerr=fake_data_err, fmt='o', color='black')  # error bars

background_cv = mc_stack[-1] - mc_stack[0]
# plt.hist(bin_centers, weights=fake_data - background_cv, bins=var_config.bins, histtype="step", color="k", label="Background")

accum_sum = [np.sum(data) for data in mc_stack]
accum_sum = [0.] + accum_sum
total_sum = accum_sum[-1]
individual_sums = [accum_sum[i + 1] - accum_sum[i] for i in range(len(accum_sum) - 1)]
fractions = [(count / total_sum) * 100 for count in individual_sums]
legend_labels = [f"{label} ({frac:.1f}%)" for label, frac in zip(mode_labels[::-1], fractions[::-1])]
legend_labels.append("Fake Data (Np x{})".format(sig_scale))
plt.legend(legend_labels, loc='upper left', fontsize=10, frameon=False, ncol=3, bbox_to_anchor=(0.05, 0.98))

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[1])
plt.ylim(0., 1.3 * fake_data.max())
plt.ylabel("Events")

if save_fig:
    plt.savefig("{}/{}-topology_breakdown.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
# inspect GENIE mode breakdown
plt.figure(figsize=(8, 6))
mc_stack, _, _ = plt.hist(var_per_genie_mode_mc,
                            bins=var_config.bins,
                            weights=weights_per_genie_mode,
                            stacked=True,
                            color=genie_mode_colors,
                            label=genie_mode_labels,
                            edgecolor='none',
                            linewidth=0,
                            density=False,
                            histtype='stepfilled')

totmc, bin_edges = np.histogram(var_total_mc, bins=var_config.bins, weights=weights_total_mc * weights_fake_data)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

# use MC as fake data for closure test
fake_data = totmc
fake_data_err = np.sqrt(totmc)
# plt.step(bin_edges[:-1], fake_data, where='post', label="Data")  # line
plt.errorbar(bin_centers, fake_data, yerr=fake_data_err, fmt='o', color='black')  # error bars

accum_sum = [np.sum(data) for data in mc_stack]
accum_sum = [0.] + accum_sum
total_sum = accum_sum[-1]
individual_sums = [accum_sum[i + 1] - accum_sum[i] for i in range(len(accum_sum) - 1)]
fractions = [(count / total_sum) * 100 for count in individual_sums]
legend_labels = [f"{label} ({frac:.1f}%)" for label, frac in zip(genie_mode_labels[::-1], fractions[::-1])]
legend_labels.append("Fake Data (Np x{})".format(sig_scale))
plt.legend(legend_labels, loc='upper left', fontsize=10, frameon=False, ncol=3, bbox_to_anchor=(0.05, 0.98))

plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[1])
plt.ylim(0., 1.3 * fake_data.max())
plt.ylabel("Events")

if save_fig:
    plt.savefig("{}/{}-genie_mode_breakdown.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
# fake data test -- MC stat + xsec cov
C_type = 2
Norm_type = 0.5
Measured = fake_data * XSEC_UNIT - background_cv * XSEC_UNIT
Model = nevts_signal_truth * XSEC_UNIT

# use MC stat and xsec cov
Covariance_Frac = ret_genie["Covariance_Frac"] + ret_mcstat["Covariance_Frac"]

Covariance = np.zeros_like(Covariance_Frac)
for i in range(len(var_config.bins)-1):
    for j in range(len(var_config.bins)-1):
        Covariance[i, j] = Covariance_Frac[i, j] * (nevts_signal_sel_reco[i] * nevts_signal_sel_reco[j]) * XSEC_UNIT**2

unfold = WienerSVD(Response, Model, Measured, Covariance, C_type, Norm_type)

In [ ]:
# true cross section for alt MC used asfake data
var_fakedata_signal_truth = mc_nu_df[mc_nu_df.truth.nu_categ == "CC1pi"][var_config.var_nu_col]
var_fakedata_signal_truth = np.clip(var_fakedata_signal_truth, var_config.bins[0], var_config.bins[-1] - eps)
weight_fakedata_signal_truth = np.ones(len(var_fakedata_signal_truth))
nevts_fakedata_signal_truth, _, _ = plt.hist(var_fakedata_signal_truth, bins=var_config.bins, weights=weight_fakedata_signal_truth, histtype="step", label="True Signal")
weight_fakedata_signal_truth[mc_nu_df[mc_nu_df.truth.nu_categ == "CC1pi"].truth.nu_categ == "CC1pi"] = sig_scale
nevts_fakedata_signal_truth, _, _ = plt.hist(var_fakedata_signal_truth, bins=var_config.bins, weights=weight_fakedata_signal_truth, histtype="step", label="MEC x{} True Signal".format(mec_scale))
plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
Unfold = unfold['unfold']
UnfoldCov = unfold['UnfoldCov']
Unfold_uncert = np.sqrt(np.diag(UnfoldCov))

step_handle, = plt.step(bin_edges, np.append(Unfold, Unfold[-1]), where='post', label='Unfolded', color='black')
bar_handle = plt.bar(
    bin_centers,
    2*Unfold_uncert,
    width=(bin_edges[1] - bin_edges[0]) * 1.,
    bottom=Unfold - Unfold_uncert,
    color='gray',
    alpha=0.5,
    linewidth=0,
    label='Unfolded Stat. error (box)'
)
reco_handle, = plt.plot(bin_centers, Measured, 'o', label='Reco. signal')
Model_smear = unfold['AddSmear'] @ Model
true_handle, = plt.plot(bin_centers, Model_smear, 'o', label='$A_c \\times$ True signal')
Fakedata_Model_smear = unfold['AddSmear'] @ nevts_fakedata_signal_truth*XSEC_UNIT
fake_true_handle, = plt.plot(bin_centers, Fakedata_Model_smear, 'o', label='Fake Data')

chi2_val = chi2(Unfold, Model_smear, UnfoldCov)
chi2_val_fakedata = chi2(Unfold, Fakedata_Model_smear, UnfoldCov)
# plt.text(0.95, 0.50, f"$ \\chi^2 = $ {chi2_val:.2f}", ha='right', va='center', 
#          transform=plt.gca().transAxes, fontsize=12)

# Custom order for legend
handles = [step_handle, bar_handle, reco_handle, true_handle, fake_true_handle]
labels = [
    'Unfolded',
    'Unfolded error',
    'Reco. signal',
    '$A_c \\times$Nominal Truth ($\\chi^2 = {:.2f}/{}$)'.format(chi2_val, len(var_config.bins)-1),
    '$A_c \\times$ Signal x{} Truth ($\\chi^2 = {:.2f}/{}$)'.format(sig_scale, chi2_val_fakedata, len(var_config.bins)-1),
]

plt.legend(handles, labels, fontsize=12)
plt.xlim(var_config.bins[0], var_config.bins[-1])
plt.xlabel(var_config.var_labels[0])
# plt.ylim(0., 5000.)
plt.ylabel("Events")

if save_fig:
    plt.savefig("{}/{}-unfolded_event_rates.pdf".format(save_fig_dir, var_config.var_save_name), bbox_inches='tight')
plt.show()

In [ ]:
save_fig_name = "{}/{}-{}-add_smear".format(save_fig_dir, var_config.var_save_name, syst_name)
plot_labels = [var_config.var_labels[2], var_config.var_labels[1]]
plot_heatmap(unfold["AddSmear"], "$A_c$", plot_labels=plot_labels,
             save_fig=save_fig, save_fig_name=save_fig_name)